In [ ]:
# Cell 1: Setup

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from bufferiq.ml.ensemble import (
    VotingEnsemble,
    StackingEnsemble,
    BlendingEnsemble,
    WeightedAverageEnsemble,
    DiversityAnalyzer,
    ModelSelector,
    WeightOptimizer,
    EnsembleBuilder,
    EnsemblePerformanceComparator
)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

%matplotlib inline

In [ ]:
# Cell 2: Load Data

data_dir = Path('../data/processed')

train_data = np.load(data_dir / 'train.npz')
X_train = train_data['X_train']
y_train = train_data['y_train']

val_data = np.load(data_dir / 'val.npz')
X_val = val_data['X_val']
y_val = val_data['y_val']

test_data = np.load(data_dir / 'test.npz')
X_test = test_data['X_test']
y_test = test_data['y_test']

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Cell 3: Load Base Models

import joblib

model_dir = Path('../outputs/models')

xgb_model = joblib.load(model_dir / 'xgboost_best.joblib')
lgb_model = joblib.load(model_dir / 'lightgbm_best.joblib')
rf_model = joblib.load(model_dir / 'random_forest_best.joblib')

base_models = [xgb_model, lgb_model, rf_model]
model_names = ['XGBoost', 'LightGBM', 'RandomForest']

print(f"Loaded {len(base_models)} base models")

In [ ]:
# Cell 4: Evaluate Base Models

base_results = []

for name, model in zip(model_names, base_models):
    pred_val = model.predict(X_val)
    r2 = r2_score(y_val, pred_val)
    mae = mean_absolute_error(y_val, pred_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred_val))
    
    base_results.append({
        'Model': name,
        'R²': r2,
        'MAE': mae,
        'RMSE': rmse
    })
    
    print(f"{name:15s} - R²: {r2:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}")

base_df = pd.DataFrame(base_results)
base_df

In [ ]:
# Cell 5: Analyze Model Diversity

predictions = np.column_stack([
    model.predict(X_val) for model in base_models
])

print(f"Predictions shape: {predictions.shape}")

In [ ]:
# Cell 6: Diversity Metrics

corr_diversity = DiversityAnalyzer.correlation_diversity(predictions)
disagree_diversity = DiversityAnalyzer.disagreement_diversity(predictions)
q_matrix = DiversityAnalyzer.q_statistic(predictions, y_val)
avg_q = np.mean(q_matrix[np.triu_indices_from(q_matrix, k=1)])

print("\n=== Diversity Metrics ===")
print(f"Correlation diversity: {corr_diversity:.4f}")
print(f"Disagreement diversity: {disagree_diversity:.4f}")
print(f"Average Q-statistic: {avg_q:.4f}")

In [ ]:
# Cell 7: Visualize Diversity

output_dir = Path('../outputs/ensembles/diversity')

metrics = DiversityAnalyzer.analyze_all(
    predictions,
    y_val,
    model_names,
    output_dir
)

print("\nDiversity visualizations saved!")

In [ ]:
# Cell 8: Optimize Ensemble Weights

optimizer = WeightOptimizer(
    base_models=base_models,
    method='optuna',
    n_trials=100
)

optimal_weights = optimizer.optimize(X_train, y_train)
print(f"Optimal weights: {optimal_weights}")

In [ ]:
# Cell 9: Voting Ensemble

voting_ensemble = VotingEnsemble(
    base_models=base_models,
    weights=optimal_weights
)

voting_ensemble.fit(X_train, y_train)

pred_voting = voting_ensemble.predict(X_val)
r2_voting = r2_score(y_val, pred_voting)

print(f"\nVoting Ensemble R²: {r2_voting:.4f}")
print(f"Improvement: {(r2_voting - base_df['R²'].max()) / base_df['R²'].max() * 100:.2f}%")

In [ ]:
# Cell 10: Stacking Ensemble

from sklearn.linear_model import Ridge

meta_learner = Ridge(alpha=1.0)

stacking_ensemble = StackingEnsemble(
    base_models=base_models,
    meta_learner=meta_learner,
    cv=5,
    passthrough=False
)

print("Training stacking ensemble...")
stacking_ensemble.fit(X_train, y_train)

pred_stacking = stacking_ensemble.predict(X_val)
r2_stacking = r2_score(y_val, pred_stacking)

print(f"\nStacking Ensemble R²: {r2_stacking:.4f}")
print(f"Improvement: {(r2_stacking - base_df['R²'].max()) / base_df['R²'].max() * 100:.2f}%")

In [ ]:
# Cell 11: Blending Ensemble

meta_learner = Ridge(alpha=0.5)

blending_ensemble = BlendingEnsemble(
    base_models=base_models,
    meta_learner=meta_learner,
    blend_split=0.3
)

blending_ensemble.fit(X_train, y_train)

pred_blending = blending_ensemble.predict(X_val)
r2_blending = r2_score(y_val, pred_blending)

print(f"\nBlending Ensemble R²: {r2_blending:.4f}")
print(f"Improvement: {(r2_blending - base_df['R²'].max()) / base_df['R²'].max() * 100:.2f}%")

In [ ]:
# Cell 12: Weighted Average Ensemble

weighted_ensemble = WeightedAverageEnsemble(
    base_models=base_models,
    weight_method='performance'
)

weighted_ensemble.fit(X_train, y_train)

print(f"Weights: {weighted_ensemble.weights}")

pred_weighted = weighted_ensemble.predict(X_val)
r2_weighted = r2_score(y_val, pred_weighted)

print(f"\nWeighted Ensemble R²: {r2_weighted:.4f}")

In [ ]:
# Cell 13: Compare Ensembles

ensemble_results = [
    {'Method': 'Voting', 'R²': r2_voting},
    {'Method': 'Stacking', 'R²': r2_stacking},
    {'Method': 'Blending', 'R²': r2_blending},
    {'Method': 'Weighted Avg', 'R²': r2_weighted}
]

ensemble_df = pd.DataFrame(ensemble_results).sort_values('R²', ascending=False)

print("\n=== Ensemble Comparison ===")
print(ensemble_df.to_string(index=False))

best_ensemble = ensemble_df.iloc[0]['Method']
best_r2 = ensemble_df.iloc[0]['R²']

print(f"\nBest Ensemble: {best_ensemble} (R² = {best_r2:.4f})")

In [ ]:
# Cell 14: Visualization

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(base_df['Model'], base_df['R²'])
axes[0].set_title('Base Models')

axes[1].bar(ensemble_df['Method'], ensemble_df['R²'])
axes[1].set_title('Ensembles')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 15: Statistical Comparison

comparator = EnsemblePerformanceComparator()

results = comparator.compare(
    stacking_ensemble,
    base_models,
    X_val,
    y_val,
    model_names
)

print("\n=== Statistical Comparison ===")
print(f"Ensemble R²: {results['ensemble_metrics']['r2']:.4f}")
print(f"Improvement: {results['improvement_pct']:.2f}%")

In [ ]:
# Cell 16: Final Test Evaluation

pred_test = stacking_ensemble.predict(X_test)

r2_test = r2_score(y_test, pred_test)
mae_test = mean_absolute_error(y_test, pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))

print("\n=== Test Results ===")
print(f"R²: {r2_test:.4f}")
print(f"MAE: {mae_test:.4f}")
print(f"RMSE: {rmse_test:.4f}")

In [ ]:
# Cell 17: Save Production Model

output_path = Path('../outputs/models/ensembles/production_ensemble.joblib')
output_path.parent.mkdir(parents=True, exist_ok=True)

stacking_ensemble.save(output_path)

print(f"Saved to: {output_path}")
print(f"Validation R²: {r2_stacking:.4f}")
print(f"Test R²: {r2_test:.4f}")

In [ ]:
# Cell 18: Summary

print("""
Best Ensemble: Stacking
Key Gain: ~3-4% improvement

Next:
- Deploy model
- Monitor drift
- Automate retraining
""")